<img src="https://raw.githubusercontent.com/AllenSWDB/allenswdb.github.io/main/databook/resources/swdb_logo_new.jpg">  

# Extension 1: Power Curves, Effect Size and Sample Size Sweeps



<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

*Building on Part 3 (a simulated real effect) and the valid workflows from Part 2 / Extension 3.*

Here we measure **power** — the true-positive rate on real-effect data — across two parameter sweeps:

1. **Effect-size sweep** — $k$ from 0.1 to 1.0 in steps of 0.1, with $N = 20$ trial groups fixed.
   Shows the minimum effect size each method can reliably detect.

2. **Sample-size sweep** — the number of trial groups $N$ from 5 to 25 in steps of 1, with
   $k = 0.2$ fixed.  Shows how many trial groups each method needs to reach adequate power.

For each combination, `N_iter_power` real-effect datasets are simulated and the
fraction of runs that produce a detection is recorded.  $P$ and the cell activity are
resampled fresh each run so the estimate reflects genuine variability.

> **Note on the circular method:** its "power" includes spurious detections arising
> from the circular selection bias — the high curve is not evidence of a trustworthy
> method, but of an inflated false-positive rate carrying over into real-effect data.

This notebook is self-contained: `loo_cv_PRCA` (Extension 3) and `fdr_bh_cells` (Part 2) are
imported from `utils.py`.

</div>


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from utils import loo_cv_PRCA, fdr_bh_cells

# Set a seed so the notebook is reproducible (remove for fresh random draws)
rng = np.random.default_rng(0)

# Simulation constants (same as Parts 1–3)
Ngroups, Ncells = 20, 200
mean_perf, sd_perf, sd_dff = 75, 8, 1

# 10% of cells are genuinely modulated by performance (the real-effect ground truth)
Nresponsive = round(0.1 * Ncells)
CorrelatedCells = np.arange(0, Nresponsive)


In [2]:
def power_sweep(param_values, param_name, N_iter, alpha, k_fixed=0.2, N_fixed=20):
    """Estimate detection power for each value of k (effect size) or N (number of trial groups).

    Simulates real-effect data for each parameter value and returns the fraction
    of runs where each method correctly reaches a detection at level alpha.

    Parameters
    ----------
    param_values : array-like
    param_name   : 'k' or 'N'
    N_iter       : simulations per parameter value
    alpha        : significance threshold
    k_fixed      : effect size used when sweeping N
    N_fixed      : number of trial groups used when sweeping k
    """
    power = {m: [] for m in ("circular", "loo_cv", "fdr_bh")}

    for val in param_values:
        k_val = float(val) if param_name == "k" else k_fixed
        N_val = int(val)   if param_name == "N" else N_fixed
        counts = {"circular": 0, "loo_cv": 0, "fdr_bh": 0}

        for _ in range(N_iter):
            # Fresh performance scores for this run
            Perf_sim = np.round(sd_perf * rng.standard_normal(N_val) + mean_perf)
            std_P = np.std(Perf_sim, ddof=1)
            if std_P < 1e-10:
                continue

            # Real-effect cell data: performance-correlated cells modulated by performance
            Pscaled_sim = (k_val * sd_dff / std_P) * (Perf_sim - np.mean(Perf_sim))
            DA_sim = sd_dff * rng.standard_normal((N_val, Ncells))
            DA_sim[:, CorrelatedCells] += Pscaled_sim[:, np.newaxis]

            # Circular
            res_s = stats.pearsonr(DA_sim, Perf_sim[:, np.newaxis], axis=0)
            sel_s = np.where((res_s.statistic > 0.1) & (res_s.pvalue < 0.05))[0]
            if sel_s.size > 0:
                _, p_circ = stats.pearsonr(DA_sim[:, sel_s].mean(axis=1), Perf_sim)
                if p_circ < alpha:
                    counts["circular"] += 1

            # LOO-CV
            PRCA_cv_s, _ = loo_cv_PRCA(DA_sim, Perf_sim)
            cv_mask = ~np.isnan(PRCA_cv_s)
            if cv_mask.sum() >= 3:
                _, p_cv = stats.pearsonr(PRCA_cv_s[cv_mask], Perf_sim[cv_mask])
                if p_cv < alpha:
                    counts["loo_cv"] += 1

            # FDR-BH: detection if >=1 cell survives BH at q = alpha
            if fdr_bh_cells(DA_sim, Perf_sim, q=alpha).size > 0:
                counts["fdr_bh"] += 1

        for m in power:
            power[m].append(counts[m] / N_iter)

    return power


In [ ]:
N_iter_power = 200   # increase to 500–1000 for smoother curves (takes longer)
alpha_power  = 0.05  # standard threshold for power analysis

k_values = np.round(np.arange(0.1, 1.01, 0.1), 2)   # [0.1, 0.2, ..., 1.0]
N_values = np.arange(5, 26, 1)                        # [5, 6, ..., 25] trial groups

print(f"Running effect-size sweep ({len(k_values)} k values × {N_iter_power} iterations) …")
power_k = power_sweep(k_values, "k", N_iter_power, alpha_power, N_fixed=20)
print("  done.")

print(f"Running sample-size sweep ({len(N_values)} N values × {N_iter_power} iterations) …")
power_N = power_sweep(N_values, "N", N_iter_power, alpha_power, k_fixed=0.2)
print("  done.")


Running effect-size sweep (10 k values × 200 iterations) …


In [ ]:
method_styles = {
    "circular": ("tab:red",    "Circular (double-dipping)"),
    "loo_cv":   ("tab:green",  "LOO-CV"),
    "fdr_bh":   ("tab:purple", "FDR-BH (corrected)"),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: power vs effect size k (N=20 trial groups fixed)
ax = axes[0]
for m, (color, label) in method_styles.items():
    ax.plot(k_values, power_k[m], "o-", color=color, label=label, linewidth=2)
ax.axhline(alpha_power, color="k", linestyle="--", linewidth=1.2,
           label=f"α = {alpha_power} (nominal FP rate)")
ax.set_xlabel("Effect size  k")
ax.set_ylabel("Detection rate (power)")
ax.set_title(f"Power vs Effect Size\n(N = 20 trial groups,  {N_iter_power} simulations per k,  α = {alpha_power})")
ax.set_xlim(k_values[0] - 0.05, k_values[-1] + 0.05)
ax.set_ylim(-0.02, 1.05)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right: power vs number of trial groups N (k=0.2 fixed)
ax = axes[1]
for m, (color, label) in method_styles.items():
    ax.plot(N_values, power_N[m], "o-", color=color, label=label, linewidth=2)
ax.axhline(alpha_power, color="k", linestyle="--", linewidth=1.2,
           label=f"α = {alpha_power} (nominal FP rate)")
ax.set_xlabel("Number of trial groups  N")
ax.set_ylabel("Detection rate (power)")
ax.set_title(f"Power vs Number of Trial Groups\n(k = 0.2,  {N_iter_power} simulations per N,  α = {alpha_power})")
ax.set_xlim(N_values[0] - 0.5, N_values[-1] + 0.5)
ax.set_ylim(-0.02, 1.05)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
